# Airline Reviews NLP Pipeline

Union two review datasets, clean and transform text, and run NLP analysis for later modeling.

Phases:
- Phase 1: collecte des Données
- Phase 2: preprocessing, tokenization, and vectorization
- Phase 3: sentiment, topics, and NER

In [2]:
# 0) Install Required Libraries (and spaCy English model)
import subprocess, sys

packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "seaborn",
    "wordcloud",
    "nltk",
    "spacy",
    "transformers",
    "torch",
    "requests",
    "beautifulsoup4",
]

def pip_install(*args):
    """Install with --user to avoid system-Python permission errors on Windows."""
    return subprocess.run(
        [sys.executable, "-m", "pip", "install", *args, "--user", "--quiet"],
        capture_output=True, text=True,
    )

for pkg in packages:
    result = pip_install(pkg)
    status = "OK" if result.returncode == 0 else f"FAILED: {result.stderr.strip()[:120]}"
    print(f"  {pkg:25s} {status}")

# spaCy English model (required for NER)
spacy_model_check = subprocess.run(
    [sys.executable, "-c", "import spacy; spacy.load('en_core_web_sm')"],
    capture_output=True, text=True,
)
if spacy_model_check.returncode != 0:
    print("  Installing spaCy model en_core_web_sm ...")
    r = subprocess.run(
        [sys.executable, "-m", "spacy", "download", "en_core_web_sm", "--user"],
        capture_output=True, text=True,
    )
    msg = "OK" if r.returncode == 0 else f"FAILED: {r.stderr.strip()[:120]}"
    print(f"  {'en_core_web_sm':25s} {msg}")
else:
    print(f"  {'en_core_web_sm':25s} OK (already installed)")

print("\nAll packages processed.")


  numpy                     OK
  pandas                    OK
  scikit-learn              OK
  scipy                     OK
  matplotlib                OK
  seaborn                   OK
  wordcloud                 OK
  nltk                      OK
  spacy                     OK
  transformers              OK
  torch                     OK
  requests                  OK
  beautifulsoup4            OK
  en_core_web_sm            OK (already installed)

All packages processed.


In [3]:
# 1) Import Libraries and Set Paths
from pathlib import Path
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings("ignore")

# ----- NLP libs (must succeed; cell 0 installs them) -----
import nltk
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download("stopwords", quiet=True)
nltk.download("vader_lexicon", quiet=True)
nltk.download("punkt", quiet=True)

import spacy
from transformers import pipeline, AutoTokenizer, AutoModel
import torch

print("Torch CUDA available:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else -1

RAW_PATH         = Path(r"C:\Users\moham\SkyInsight\10_NLP\airline_reviews_raw_data.csv")
SCRAPED_PATH     = Path(r"C:\Users\moham\SkyInsight\10_NLP\airline_reviews_scraped.csv")
WEB_SCRAPED_PATH = Path(r"C:\Users\moham\SkyInsight\10_NLP\airline_reviews_web_scraped.csv")
OUTPUT_DIR       = Path(r"C:\Users\moham\SkyInsight\10_NLP\outputs_nlp")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output:", OUTPUT_DIR)


Torch CUDA available: False
Output: C:\Users\moham\SkyInsight\10_NLP\outputs_nlp


In [4]:
# 2) Load Raw, Scraped, and Web-Scraped CSV Files
def safe_read_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path, encoding="utf-8", encoding_errors="ignore")

df_raw         = safe_read_csv(RAW_PATH)
df_scraped     = safe_read_csv(SCRAPED_PATH)
df_web_scraped = safe_read_csv(WEB_SCRAPED_PATH)

print("Raw shape:",         df_raw.shape)
print("Scraped shape:",     df_scraped.shape)
print("Web-scraped shape:", df_web_scraped.shape)
display(df_raw.head(3))
display(df_scraped.head(3))
display(df_web_scraped.head(3))
print("Raw columns:",         df_raw.columns.tolist())
print("Scraped columns:",     df_scraped.columns.tolist())
print("Web-scraped columns:", df_web_scraped.columns.tolist())


Raw shape: (5000, 6)
Scraped shape: (897, 6)
Web-scraped shape: (750, 6)


,airline,source,review_text,rating,date,verified_purchase
0,Talyena Airlines,repo_satisfaction_ratings,Seat Comfort was excellent. Food and Drink was...,2,2026-05-02,yes
1,Talyena Airlines,repo_satisfaction_ratings,Seat Comfort was good. Food and Drink was aver...,4,2026-05-02,yes
2,Talyena Airlines,repo_satisfaction_ratings,Seat Comfort was excellent. Food and Drink was...,4,2026-05-02,yes


,airline,source,review_text,rating,date,verified_purchase
0,Lufthansa,pullpush_reddit,Airlines Are Collecting Your Data And Selling ...,4,2026-05-02,unknown
1,Southwest Airlines,pullpush_reddit,Explore Multi-City Flight Packages with Car an...,5,2026-05-02,unknown
2,Qatar Airways,pullpush_reddit,"Critical News Committee - May 7th, 2025. https...",5,2026-05-02,unknown


,airline,source,review_text,rating,date,verified_purchase
0,Emirates,skytrax,"Like others, having a complete headache gettin...",1,2026-05-01,unknown
1,Emirates,skytrax,"Due to the war situation in the Middle East, I...",1,2026-04-30,unknown
2,Emirates,skytrax,I recently booked a flight with Emirates from ...,1,2026-04-08,unknown


Raw columns: ['airline', 'source', 'review_text', 'rating', 'date', 'verified_purchase']
Scraped columns: ['airline', 'source', 'review_text', 'rating', 'date', 'verified_purchase']
Web-scraped columns: ['airline', 'source', 'review_text', 'rating', 'date', 'verified_purchase']


In [5]:
# 3) Union the Datasets (raw + reddit-scraped + web-scraped) and Save Combined CSV
all_cols = sorted(
    set(df_raw.columns)
    .union(set(df_scraped.columns))
    .union(set(df_web_scraped.columns))
)
df_raw_aligned         = df_raw.reindex(columns=all_cols)
df_scraped_aligned     = df_scraped.reindex(columns=all_cols)
df_web_scraped_aligned = df_web_scraped.reindex(columns=all_cols)

df_all = pd.concat(
    [df_raw_aligned, df_scraped_aligned, df_web_scraped_aligned],
    ignore_index=True
)
before_dupes = len(df_all)
df_all = df_all.drop_duplicates()
after_dupes = len(df_all)

combined_path = OUTPUT_DIR / "airline_reviews_combined.csv"
df_all.to_csv(combined_path, index=False)

print(f"Sources: raw={len(df_raw)}, reddit={len(df_scraped)}, web={len(df_web_scraped)}")
print(f"Combined rows: {before_dupes} -> {after_dupes} (after dedup)")
print(f"Saved: {combined_path}")


Sources: raw=5000, reddit=897, web=750
Combined rows: 6647 -> 5291 (after dedup)
Saved: C:\Users\moham\SkyInsight\10_NLP\outputs_nlp\airline_reviews_combined.csv


In [6]:
# 4) Text Cleaning: Normalize, Remove Duplicates and Stopwords
import re
import pandas as pd
import nltk

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

def detect_text_column(columns):
    candidates = ["review", "review_text", "text", "content", "comment", "comments", "feedback", "body", "title"]
    cols_lower = {c.lower(): c for c in columns}
    for c in candidates:
        if c in cols_lower:
            return cols_lower[c]
    for c in columns:
        if any(k in c.lower() for k in candidates):
            return c
    return None

text_col = detect_text_column(df_all.columns)
if text_col is None:
    raise ValueError("Could not detect a review text column.")
print("Text column:", text_col)

stop_words = set(stopwords.words("english"))

def clean_text(s):
    s = "" if pd.isna(s) else str(s)
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df_all["text_clean"] = df_all[text_col].apply(clean_text)
df_all = df_all.drop_duplicates(subset=["text_clean"]).reset_index(drop=True)

def remove_stopwords(s):
    tokens = [t for t in s.split() if t not in stop_words]
    return " ".join(tokens)

df_all["text_nostop"] = df_all["text_clean"].apply(remove_stopwords)
print("Rows after cleaning:", len(df_all))


Text column: review_text
Rows after cleaning: 5190


In [ ]:
# 5) Tokenization and Vectorization (TF-IDF + DistilBERT Embeddings)
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel
import torch
import json
import numpy as np

# ---- TF-IDF ----
tfidf = TfidfVectorizer(max_features=4000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df_all["text_nostop"])

tfidf_path = OUTPUT_DIR / "tfidf_matrix.npz"
vocab_path = OUTPUT_DIR / "tfidf_vocab.json"
sparse.save_npz(tfidf_path, tfidf_matrix)
with open(vocab_path, "w", encoding="utf-8") as f:
    # Cast numpy int64 values to plain Python int before serializing
    json.dump({k: int(v) for k, v in tfidf.vocabulary_.items()}, f)
print(f"TF-IDF matrix: {tfidf_matrix.shape} -> {tfidf_path}")

# ---- DistilBERT embeddings (required, not skipped) ----
embeddings_path = OUTPUT_DIR / "text_embeddings.npy"

DEVICE = 0 if torch.cuda.is_available() else -1

emb_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
emb_model     = AutoModel.from_pretrained("distilbert-base-uncased")
emb_model.eval()
if DEVICE == 0:
    emb_model = emb_model.to("cuda")

def mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    masked = last_hidden * mask
    return masked.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

texts = df_all["text_clean"].fillna("").astype(str).tolist()
batch_size = 16
all_embeds = []
for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    tok = emb_tokenizer(
        batch, padding=True, truncation=True, max_length=256, return_tensors="pt"
    )
    if DEVICE == 0:
        tok = {k: v.to("cuda") for k, v in tok.items()}
    with torch.no_grad():
        outputs = emb_model(**tok)
    embeds = mean_pool(outputs.last_hidden_state, tok["attention_mask"]).cpu().numpy()
    all_embeds.append(embeds)
    if (i // batch_size) % 25 == 0:
        print(f"  embeddings: {min(i + batch_size, len(texts))}/{len(texts)}")

all_embeds = np.vstack(all_embeds)
np.save(embeddings_path, all_embeds)
print(f"DistilBERT embeddings: {all_embeds.shape} -> {embeddings_path}")


TF-IDF matrix: (5190, 4000) -> C:\Users\moham\SkyInsight\10_NLP\outputs_nlp\tfidf_matrix.npz


In [ ]:
# 6) Sentiment Analysis (VADER + DistilBERT) — full coverage, no skipping
import pandas as pd
import numpy as np
import nltk
import torch
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1

# VADER: lexicon-based, runs on every row
sia = SentimentIntensityAnalyzer()
df_all["sent_vader"] = df_all["text_clean"].apply(
    lambda s: sia.polarity_scores(s if isinstance(s, str) else "")["compound"]
)
df_all["sent_vader_label"] = pd.cut(
    df_all["sent_vader"],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=["negative", "neutral", "positive"],
)
print("VADER distribution:")
print(df_all["sent_vader_label"].value_counts())

# DistilBERT fine-tuned on SST-2 — runs on ALL rows with truncation to avoid >512 token errors
sent_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=DEVICE,
    truncation=True,
    max_length=512,
)

texts = df_all["text_clean"].fillna("").astype(str).tolist()
labels, scores = [], []
batch = 32
for i in range(0, len(texts), batch):
    out = sent_pipe(texts[i:i + batch])
    labels.extend(r["label"] for r in out)
    scores.extend(r["score"] for r in out)
    if (i // batch) % 20 == 0:
        print(f"  DistilBERT sentiment: {min(i + batch, len(texts))}/{len(texts)}")

df_all["sent_bert_label"] = labels
df_all["sent_bert_score"] = scores
print("\nDistilBERT distribution:")
print(df_all["sent_bert_label"].value_counts())


DistilBERT skipped: transformers not available.


In [ ]:
# 7) Topic Modeling with LDA
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd

count_vec = CountVectorizer(max_features=3000, stop_words="english")
dtm = count_vec.fit_transform(df_all["text_nostop"])

n_topics = 8
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
topic_matrix = lda.fit_transform(dtm)
df_all["topic_id"] = topic_matrix.argmax(axis=1)

terms = np.array(count_vec.get_feature_names_out())
topics = []
for k, comp in enumerate(lda.components_):
    top_idx = comp.argsort()[::-1][:10]
    topics.append({"topic_id": k, "top_terms": ", ".join(terms[top_idx])})

topics_df = pd.DataFrame(topics)
topics_path = OUTPUT_DIR / "lda_topics.csv"
topics_df.to_csv(topics_path, index=False)
print(f"Saved topics: {topics_path}")
print(topics_df.to_string(index=False))


Saved topics: C:\Users\moham\SkyInsight\10_NLP\outputs_nlp\lda_topics.csv


In [ ]:
# 8) Named Entity Recognition (NER) with spaCy on ALL rows
import spacy
import pandas as pd
from collections import Counter

nlp_ner = spacy.load("en_core_web_sm", disable=["parser", "tagger", "lemmatizer"])

texts_all = df_all["text_clean"].fillna("").astype(str).tolist()
entities_col = []
for doc in nlp_ner.pipe(texts_all, batch_size=64):
    ents = sorted({f"{ent.text}:{ent.label_}" for ent in doc.ents if ent.text.strip()})
    entities_col.append(", ".join(ents))

df_all["entities"] = entities_col

# Aggregate top entities by label
ent_counter = Counter()
for cell in entities_col:
    if not cell:
        continue
    for token in cell.split(", "):
        if ":" in token:
            ent_counter[token] += 1

top_entities = pd.DataFrame(ent_counter.most_common(50), columns=["entity", "count"])
top_entities[["text", "label"]] = top_entities["entity"].str.rsplit(":", n=1, expand=True)
ner_path = OUTPUT_DIR / "top_entities.csv"
top_entities.to_csv(ner_path, index=False)
print(f"NER complete. Saved top entities: {ner_path}")
print(top_entities.head(15).to_string(index=False))


NER skipped: no spaCy or transformers available.


In [ ]:
# 9) Export Preprocessed Data and NLP Outputs
import json
from pathlib import Path

cleaned_path = OUTPUT_DIR / "airline_reviews_cleaned.csv"
df_all.to_csv(cleaned_path, index=False)

summary = {
    "rows": int(len(df_all)),
    "columns": df_all.columns.tolist(),
    "text_column": text_col,
    "tfidf_matrix_path": str(tfidf_path),
    "embeddings_path": str(embeddings_path),
    "topics_path": str(topics_path),
    "ner_path": str(ner_path),
    "vader_distribution": {str(k): int(v) for k, v in df_all["sent_vader_label"].value_counts().items()},
    "bert_distribution":  {str(k): int(v) for k, v in df_all["sent_bert_label"].value_counts().items()},
}
summary_path = OUTPUT_DIR / "nlp_export_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Saved cleaned data: {cleaned_path}")
print(f"Saved summary:      {summary_path}")


Saved cleaned data: C:\Users\moham\SkyInsight\10_NLP\outputs_nlp\airline_reviews_cleaned.csv
Saved summary: C:\Users\moham\SkyInsight\10_NLP\outputs_nlp\nlp_export_summary.json


In [ ]:
# 10) Visualization & Validation (Phase 3 / Phase 5 deliverables)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sentiment distributions
vader_counts = df_all["sent_vader_label"].value_counts()
vader_counts.plot(kind="bar", ax=axes[0, 0], color=["#d9534f", "#f0ad4e", "#5cb85c"][:len(vader_counts)])
axes[0, 0].set_title("VADER Sentiment Distribution")
axes[0, 0].set_ylabel("Reviews")
axes[0, 0].tick_params(axis="x", rotation=0)

bert_counts = df_all["sent_bert_label"].value_counts()
bert_counts.plot(kind="bar", ax=axes[0, 1], color=["#d9534f", "#5cb85c"][:len(bert_counts)])
axes[0, 1].set_title("DistilBERT Sentiment Distribution")
axes[0, 1].set_ylabel("Reviews")
axes[0, 1].tick_params(axis="x", rotation=0)

# Topic distribution
df_all["topic_id"].value_counts().sort_index().plot(kind="bar", ax=axes[1, 0], color="#337ab7")
axes[1, 0].set_title("LDA Topic Distribution")
axes[1, 0].set_xlabel("topic_id")

# VADER score histogram
axes[1, 1].hist(df_all["sent_vader"].dropna(), bins=30, color="#5bc0de", edgecolor="black")
axes[1, 1].set_title("VADER Compound Score Histogram")
axes[1, 1].set_xlabel("compound score")

plt.tight_layout()
viz_path = OUTPUT_DIR / "nlp_overview.png"
plt.savefig(viz_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved overview figure: {viz_path}")

# Cross-check VADER vs DistilBERT agreement (Phase 5 validation)
def vader_to_binary(lbl):
    if lbl == "positive":
        return "POSITIVE"
    if lbl == "negative":
        return "NEGATIVE"
    return None

mask = df_all["sent_vader_label"].apply(vader_to_binary).notna()
agree = (
    df_all.loc[mask, "sent_vader_label"].apply(vader_to_binary)
    == df_all.loc[mask, "sent_bert_label"]
).mean()
print(f"VADER vs DistilBERT agreement (excluding neutral): {agree:.2%}")
